# AI Programming — Lecture 23
## Lab 1: MNIST Image Generation with DCGAN

이번 실습에서는 **DCGAN (Deep Convolutional GAN)**의 핵심 구조와 학습 과정을 직접 구현합니다.

```text
Random latent vector z
        ↓
    Generator
        ↓
   Fake image
        ↓
  Discriminator
        ↓
   Real / Fake
```

### 학습 목표
- Generator와 Discriminator의 역할을 설명할 수 있습니다.
- Generator에서 **transposed convolution**을 이용해 이미지를 upsampling합니다.
- Discriminator에서 **strided convolution**을 이용해 이미지를 판별합니다.
- GAN의 **alternating training**을 직접 구현합니다.
- Generator에 **non-saturating loss**를 사용합니다.
- Training 중 생성 이미지가 어떻게 변하는지 관찰합니다.
- Latent space interpolation을 수행합니다.

> 이번 실습의 목표는 최고 품질의 MNIST 이미지를 만드는 것보다 **GAN 학습 원리를 이해하는 것**입니다.

## 0. 실습 환경 설정

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
LATENT_DIM = 100
BATCH_SIZE = 256
EPOCHS = 10

tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus[0].name if gpus else "사용하지 않음 (CPU)")

## 1. MNIST Dataset

Generator의 마지막 activation으로 `tanh`를 사용하므로 real image도 $[-1,1]$ 범위로 scaling합니다.

In [ ]:
(x_train, _), (_, _) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32")
x_train = (x_train - 127.5) / 127.5
x_train = np.expand_dims(x_train, axis=-1)

print("Training data:", x_train.shape)
print("Pixel range:", x_train.min(), "~", x_train.max())

train_dataset = (
    tf.data.Dataset
    .from_tensor_slices(x_train)
    .shuffle(len(x_train), seed=SEED)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.AUTOTUNE)
)

## 2. Generator

```text
z ∈ R^100
↓
Dense
↓
7 × 7 × 128
↓
Conv2DTranspose
↓
14 × 14 × 64
↓
Conv2DTranspose
↓
28 × 28 × 1
```

DCGAN에서는 pooling 대신 **transposed convolution**으로 spatial resolution을 키웁니다.

In [ ]:
def build_generator():
    model = keras.Sequential(name="Generator")

    model.add(layers.Input(shape=(LATENT_DIM,)))

    model.add(
        layers.Dense(
            7 * 7 * 128,
            use_bias=False
        )
    )
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    model.add(layers.Reshape((7, 7, 128)))

    model.add(
        layers.Conv2DTranspose(
            64,
            kernel_size=5,
            strides=2,
            padding="same",
            use_bias=False
        )
    )
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    model.add(
        layers.Conv2DTranspose(
            1,
            kernel_size=5,
            strides=2,
            padding="same",
            activation="tanh"
        )
    )

    return model

generator = build_generator()
generator.summary()

### 학습 전 Generator 출력

In [ ]:
fixed_noise = tf.random.normal(
    [16, LATENT_DIM],
    seed=SEED
)

generated = generator(
    fixed_noise,
    training=False
)

plt.figure(figsize=(4, 4))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(
        generated[i, :, :, 0],
        cmap="gray",
        vmin=-1,
        vmax=1
    )
    plt.axis("off")

plt.suptitle("Before Training")
plt.tight_layout()
plt.show()

## 3. Discriminator

```text
28 × 28 × 1
↓
Strided Conv
↓
14 × 14 × 64
↓
Strided Conv
↓
7 × 7 × 128
↓
Dense
↓
Real / Fake
```

DCGAN에서는 pooling 대신 **strided convolution**을 사용합니다.

In [ ]:
def build_discriminator():
    model = keras.Sequential(name="Discriminator")

    model.add(layers.Input(shape=(28, 28, 1)))

    model.add(
        layers.Conv2D(
            64,
            kernel_size=5,
            strides=2,
            padding="same"
        )
    )
    model.add(
        layers.LeakyReLU(
            negative_slope=0.2
        )
    )
    model.add(layers.Dropout(0.3))

    model.add(
        layers.Conv2D(
            128,
            kernel_size=5,
            strides=2,
            padding="same"
        )
    )
    model.add(
        layers.LeakyReLU(
            negative_slope=0.2
        )
    )
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

discriminator = build_discriminator()
discriminator.summary()

## 4. GAN Loss

### Discriminator

```text
Real image → label 1
Fake image → label 0
```

### Generator

Generator는 fake image가 discriminator에게 **real(1)**로 판별되도록 학습합니다.

이것이 강의에서 설명한 **non-saturating generator loss**입니다.

In [ ]:
cross_entropy = keras.losses.BinaryCrossentropy(
    from_logits=True
)

def discriminator_loss(real_logits, fake_logits):
    real_loss = cross_entropy(
        tf.ones_like(real_logits),
        real_logits
    )

    fake_loss = cross_entropy(
        tf.zeros_like(fake_logits),
        fake_logits
    )

    return real_loss + fake_loss

def generator_loss(fake_logits):
    return cross_entropy(
        tf.ones_like(fake_logits),
        fake_logits
    )

## 5. Optimizer

In [ ]:
generator_optimizer = keras.optimizers.Adam(
    learning_rate=2e-4,
    beta_1=0.5
)

discriminator_optimizer = keras.optimizers.Adam(
    learning_rate=2e-4,
    beta_1=0.5
)

## 6. Alternating Training

한 training step:

```text
1. Real image 준비
2. z sampling
3. Generator가 fake image 생성
4. Discriminator: real→1, fake→0
5. Generator: fake→1이 되도록 학습
6. 두 network를 각각 update
```

In [ ]:
@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]

    noise = tf.random.normal(
        [batch_size, LATENT_DIM]
    )

    with tf.GradientTape() as gen_tape,          tf.GradientTape() as disc_tape:

        fake_images = generator(
            noise,
            training=True
        )

        real_logits = discriminator(
            real_images,
            training=True
        )

        fake_logits = discriminator(
            fake_images,
            training=True
        )

        gen_loss = generator_loss(
            fake_logits
        )

        disc_loss = discriminator_loss(
            real_logits,
            fake_logits
        )

    gen_gradients = gen_tape.gradient(
        gen_loss,
        generator.trainable_variables
    )

    disc_gradients = disc_tape.gradient(
        disc_loss,
        discriminator.trainable_variables
    )

    generator_optimizer.apply_gradients(
        zip(
            gen_gradients,
            generator.trainable_variables
        )
    )

    discriminator_optimizer.apply_gradients(
        zip(
            disc_gradients,
            discriminator.trainable_variables
        )
    )

    return gen_loss, disc_loss

## 7. 생성 이미지 시각화 함수

In [ ]:
def show_generated_images(model, noise, title):
    predictions = model(
        noise,
        training=False
    )

    plt.figure(figsize=(4, 4))

    for i in range(16):
        plt.subplot(4, 4, i + 1)
        plt.imshow(
            predictions[i, :, :, 0],
            cmap="gray",
            vmin=-1,
            vmax=1
        )
        plt.axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 8. DCGAN Training

같은 `fixed_noise`를 계속 사용하여 epoch에 따른 생성 결과 변화를 관찰합니다.

In [ ]:
gen_loss_history = []
disc_loss_history = []

for epoch in range(EPOCHS):
    epoch_gen_losses = []
    epoch_disc_losses = []

    for real_images in train_dataset:
        gen_loss, disc_loss = train_step(
            real_images
        )

        epoch_gen_losses.append(
            float(gen_loss)
        )

        epoch_disc_losses.append(
            float(disc_loss)
        )

    mean_gen_loss = np.mean(epoch_gen_losses)
    mean_disc_loss = np.mean(epoch_disc_losses)

    gen_loss_history.append(mean_gen_loss)
    disc_loss_history.append(mean_disc_loss)

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | "
        f"G loss={mean_gen_loss:.4f} | "
        f"D loss={mean_disc_loss:.4f}"
    )

    if (
        epoch == 0
        or (epoch + 1) % 5 == 0
        or epoch == EPOCHS - 1
    ):
        show_generated_images(
            generator,
            fixed_noise,
            title=f"Epoch {epoch + 1}"
        )

## 9. GAN Loss Curve

GAN loss는 일반 supervised learning처럼 매끄럽게 계속 감소하지 않을 수 있습니다.
Generator와 Discriminator가 경쟁하기 때문입니다.

따라서 **loss와 생성 이미지 품질을 함께** 확인해야 합니다.

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    gen_loss_history,
    label="Generator Loss"
)

plt.plot(
    disc_loss_history,
    label="Discriminator Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN Training")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 10. 새로운 이미지 생성

In [ ]:
new_noise = tf.random.normal(
    [25, LATENT_DIM],
    seed=123
)

new_images = generator(
    new_noise,
    training=False
)

plt.figure(figsize=(5, 5))

for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(
        new_images[i, :, :, 0],
        cmap="gray",
        vmin=-1,
        vmax=1
    )
    plt.axis("off")

plt.suptitle("Generated MNIST Samples")
plt.tight_layout()
plt.show()

## 11. Latent Space Interpolation

두 latent vector 사이를 선형 보간합니다.

$$
z(\alpha)
=
(1-\alpha)z_1
+
\alpha z_2
$$

In [ ]:
z1 = tf.random.normal(
    [1, LATENT_DIM],
    seed=111
)

z2 = tf.random.normal(
    [1, LATENT_DIM],
    seed=222
)

alphas = np.linspace(
    0.0,
    1.0,
    10
).astype("float32")

z_interp = tf.concat(
    [
        (1.0 - a) * z1 + a * z2
        for a in alphas
    ],
    axis=0
)

interp_images = generator(
    z_interp,
    training=False
)

plt.figure(figsize=(12, 2))

for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(
        interp_images[i, :, :, 0],
        cmap="gray",
        vmin=-1,
        vmax=1
    )
    plt.axis("off")

plt.suptitle("Latent Space Interpolation")
plt.tight_layout()
plt.show()

## 12. 직접 해보기

1. `EPOCHS = 5`, `10`, `20`을 비교하세요.
2. 같은 `fixed_noise`에서 epoch별 생성 결과가 어떻게 변하는지 확인하세요.
3. 다른 `z1`, `z2`로 latent interpolation을 반복해 보세요.
4. Generator와 Discriminator loss가 계속 감소하지 않는 이유를 설명하세요.
5. 비슷한 숫자만 반복해서 생성한다면 어떤 문제를 의심할 수 있을까요?

### 꼭 기억할 것

- Generator: `z → image`
- Discriminator: `image → real/fake`
- GAN은 두 network를 번갈아 학습합니다.
- DCGAN은 transposed convolution, strided convolution, Batch Normalization을 활용합니다.
- Generator에는 non-saturating loss를 사용합니다.
- GAN에는 training instability와 mode collapse가 발생할 수 있습니다.